In [ ]:
# "nbsphinx": "hidden"
import os

os.chdir("../../../tests/datasets")

# Pipeline

A `GchopPipeline` assembles several preprocessing steps (*transformers*) and a final decomposer, so that the whole analysis can be applied to a galaxy in one call, setting the parameters of every step.

Each step is a `(name, step)` pair: the transformers are applied sequentially, and the last step decomposes the result.

In [1]:
import galaxychop as gchop

gal = gchop.io.read_hdf5("gal394242.h5")
gal

,Particles,$\mathrm{M_{\odot}}$
Stars ⭐,37393,3.74e+10
Dark Matter ⚫​,155101,1.16e+12
Gas 💨,80153,1.22e+11


In [2]:
center = gchop.preproc.Centralizer()
align = gchop.preproc.Aligner()

print("Centered:", center.checker(gal))
print("Aligned:", align.checker(gal))

Centered: False
Aligned: False


The galaxy is neither centered nor aligned. Here is a pipeline that centers it, aligns it and then decomposes it with `JThreshold`.

By default `Aligner` uses only the stars within `r_cut=30` kpc to compute the rotation, so the alignment follows the disk and not the stellar halo; pass `r_cut=None` to use every star.

In [3]:
pipe = gchop.GchopPipeline(
    [
        ("center", gchop.preproc.Centralizer(with_potential=True)),
        ("align", gchop.preproc.Aligner(r_cut=30)),
        ("decompose", gchop.decomposers.JThreshold()),
    ]
)
pipe

`transform()` applies only the transformers:

In [4]:
transformed_gal = pipe.transform(gal)

print("Centered:", center.checker(transformed_gal))
print("Aligned:", align.checker(transformed_gal))

Centered: True
Aligned: True


And `decompose()` applies the transformers and then the decomposer, returning a `DecomposedGalaxy`:

In [5]:
gal_decomposed = pipe.decompose(gal)
gal_decomposed